In [22]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.tools as tools
import plotly.graph_objects as go

import ast

In [23]:
%run helpers.ipynb

In [24]:
oscar_with_imdb, imdb_non_oscar = read_data('oscar_with_imdb.csv', 'imdb_non_oscar.csv')

oscar_with_imdb['Status'] = oscar_with_imdb['Winner'].apply(lambda row: 'Winner' if row == True else 'Nominated')
imdb_non_oscar['Status'] = 'No nomination'

In [25]:
columns = ['title', 'rating', 'meta_score', 'votes', 'languages', 'Status']

oscars_df = oscar_with_imdb[columns]
imdb_df = imdb_non_oscar[columns]

graph_df = pd.concat([oscars_df[columns], imdb_df[columns]], axis=0)

In [26]:
graph_df.head()

,title,rating,meta_score,votes,languages,Status
0,The Noose,6.5,NaN,93.0,"['None', 'English']",Nominated
1,The Patent Leather Kid,6.0,NaN,623.0,"['None', 'English']",Nominated
2,The Last Command,7.9,NaN,4800.0,"['None', 'English']",Winner
3,The Way of All Flesh,6.6,NaN,241.0,"['None', 'English']",Winner
4,A Ship Comes In,5.5,NaN,246.0,['None'],Nominated


In [27]:
graph_df.dropna(subset=['languages'], inplace=True)

In [28]:
%run helpers.ipynb

In [29]:
graph_df['language'] = graph_df['languages'].apply(lambda x: get_languages(x))
graph_df = graph_df[graph_df['language'] != '']
filtered = graph_df[~graph_df['language'].isin(['English', 'None'])] # Ceck for lowercase, so far all are capitalized

In [30]:
print(len(graph_df))

63698


In [31]:
topn = 10
most_frequent_langs = (
    filtered['language']
    .value_counts()
    .nlargest(topn)
    .index
)

In [32]:
# Non English movies
len(filtered)

20657

In [33]:
top_10_df = filtered[filtered['language'].isin(most_frequent_langs)]

In [34]:
top_10_grouped = top_10_df.groupby(['language', 'Status']).agg(
    avg_rating=('rating', 'mean'),
    sum_votes=('votes', 'sum'),
    total_films=('language', 'size')
).reset_index()

In [35]:
# top_10_df.sort_values(['total_films'], ascending=False)[:topn]

In [36]:
top_10_grouped

,language,Status,avg_rating,sum_votes,total_films
0,Cantonese,No nomination,6.364667,2767840.0,450
1,Cantonese,Nominated,6.800000,24000.0,1
2,French,No nomination,6.353665,13060855.0,3315
3,French,Nominated,7.401399,8857812.0,143
4,French,Winner,7.391667,780991.0,24
5,German,No nomination,6.144858,3948897.0,1799
6,German,Nominated,7.557500,4238337.0,40
7,German,Winner,7.550000,1875434.0,12
8,Hindi,No nomination,6.414255,13700551.0,1838
9,Hindi,Nominated,7.700000,226000.0,5


In [37]:
language = top_10_grouped['language'].unique()
statuses = top_10_grouped['Status'].unique()

# Helper: Build bar traces for each metric
def make_traces(metric):
    return [
        go.Bar(
            x=top_10_grouped[top_10_grouped['Status'] == status]['language'],
            y=top_10_grouped[top_10_grouped['Status'] == status][metric],
            name=status
        )
        for status in statuses
    ]

fig = go.Figure()

# Add initial traces (e.g., count)
for trace in make_traces('avg_rating'):
    fig.add_trace(trace)

# Updatemenus for toggle buttons
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=list([
                dict(
                    label="Average Rating",
                    method="update",
                    args=[{"y": [top_10_grouped[top_10_grouped['Status'] == status]['avg_rating'].values for status in statuses]}]
                ),
                dict(
                    label="Number of Films",
                    method="update",
                    args=[{"y": [top_10_grouped[top_10_grouped['Status'] == status]['total_films'].values for status in statuses]}]
                ),
                dict(
                    label="Total Votes",
                    method="update",
                    args=[{"y": [top_10_grouped[top_10_grouped['Status'] == status]['sum_votes'].values for status in statuses]}]
                ),
            ])
        )
    ]
)

fig.update_layout(barmode='group')
fig.show()

In [38]:
fig = px.bar(
    top_10_grouped,
    x="Status",
    y="avg_rating",
    color="language",
    barmode="group",
    title="Grouped bar chart"
)

# Add dropdown buttons to allow toggling columns
fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            buttons=list([
                dict(
                    label="Average Rating",
                    method="update",
                    args=[
                        {"y": [top_10_grouped.loc[top_10_grouped['Status'] == col, 'avg_rating'] for col in top_10_grouped['Status'].unique()]},
                        {"yaxis": {"title": "Average Rating"}}
                    ]
                ),
                dict(
                    label="Total Votes",
                    method="update",
                    args=[
                        {"y": [top_10_grouped.loc[top_10_grouped['Status'] == col, 'sum_votes'] for col in top_10_grouped['Status'].unique()]},
                        {"yaxis": {"title": "Total Votes"}}
                    ]
                ),
                dict(
                    label="Number of Films",
                    method="update",
                    args=[
                        {"y": [top_10_grouped.loc[top_10_grouped['Status'] == col, 'total_films'] for col in top_10_grouped['Status'].unique()]},
                        {"yaxis": {"title": "Number of Films"}}
                    ]
                ),
            ])
        )
    ]
)

fig.show()